<div align="center">
    <span style="font-size:30px">
        <strong>
            <!-- Símbolo de Python -->
            <img
                src="https://cdn3.emoji.gg/emojis/1887_python.png"
                style="margin-bottom:-5px"
                width="30px" 
                height="30px"
            >
            <!-- Título -->
            Python para Geólogos
            <!-- Versión -->
            <img 
                src="https://img.shields.io/github/release/kevinalexandr19/manual-python-geologia.svg?style=flat&label=&color=blue"
                style="margin-bottom:-2px" 
                width="40px"
            >
        </strong>
    </span>
    <br>
    <span>
        <!-- Github del proyecto -->
        <a href="https://github.com/kevinalexandr19/manual-python-geologia" target="_blank">
            <img src="https://img.shields.io/github/stars/kevinalexandr19/manual-python-geologia.svg?style=social&label=Github Repo">
        </a>
        &nbsp;&nbsp;
        <!-- Licencia -->
        <img src="https://img.shields.io/github/license/kevinalexandr19/manual-python-geologia.svg?color=forestgreen">
        &nbsp;&nbsp;
        <!-- Release date -->
        <img src="https://img.shields.io/github/release-date/kevinalexandr19/manual-python-geologia?color=gold">
    </span>
    <br>
    <span>
        <!-- Perfil de LinkedIn -->
        <a target="_blank" href="https://www.linkedin.com/in/kevin-alexander-gomez/">
            <img src="https://img.shields.io/badge/-Kevin Alexander Gomez-5eba00?style=social&logo=linkedin">
        </a>
        &nbsp;&nbsp;
        <!-- Perfil de Github -->
        <a target="_blank" href="https://github.com/kevinalexandr19">
            <img src="https://img.shields.io/github/followers/kevinalexandr19.svg?style=social&label=kevinalexandr19&maxAge=2592000">
        </a>
    </span>
    <br>
</div>

***

<span style="color:lightgreen; font-size:25px">**PG103 - Visualización de datos en Geología**</span>

Bienvenido al curso!!!

Vamos a revisar ejemplos de <span style="color:gold">visualización de datos</span> en Geología usando código en Python. <br>
Es necesario que tengas un conocimiento previo en programación con Python, estadística y geología general.


<span style="color:gold; font-size:20px">**Visualización de un modelo de bloques en 3D**</span>

***
En geología, un <span style="color:gold">modelo de bloques</span> es una representación tridimensional de un yacimiento mineral en la que se divide el área en bloques cúbicos o rectangulares.

Cada bloque representa un volumen homogéneo dentro del yacimiento y tiene asignado un valor o conjunto de valores que indican la presencia y cantidad de minerales en ese bloque.

Los modelos de bloques se utilizan en la industria minera para estimar las reservas minerales y planificar la extracción del mineral. <br>
El proceso de creación de un modelo de bloques comienza con la recopilación de datos geológicos y de exploración del yacimiento, como datos de perforación, análisis geoquímicos y muestreo de superficie.

A partir de estos datos, se utiliza un software especializado para crear un modelo tridimensional del yacimiento dividido en bloques. <br> 
A cada bloque se asigna un valor o conjunto de valores que indican las diferentes variables de interés (ley, porosidad, densidad, etc.). <br>
Estos valores se pueden determinar a través de diferentes métodos, como la estimación por kriging, la simulación estocástica, entre otros.

Una vez que se ha creado el modelo de bloques, se realizar la estimación de reservas minerales y se planifica la extracción del mineral.

<span style="color:#43c6ac">La precisión y confiabilidad del modelo de bloques depende en gran medida de la calidad de los datos geológicos y de exploración utilizados en su creación, así como de los métodos y supuestos utilizados para asignar valores a los bloques.</span>
***

Vamos a leer cómo los centroides se convierten en superficies de bloques. El objetivo es entender **qué información contiene el modelo, cómo se representa en 3D y qué podemos interpretar de la figura**.

### <span style="color:gold">**Antes de ejecutar**</span>

En JupyterLite, ejecuta la siguiente celda y espera a que termine la preparación. En Python local, instala las librerías con `%pip install plotly==6.5.2 "nbformat>=5,<6"`. Los archivos de entrada deben estar en la carpeta `files`, junto al notebook.


In [ ]:
import sys
# Preparar las librerías del navegador antes de continuar.
if sys.platform == "emscripten":
    import piplite
    await piplite.install(['plotly==6.5.2', 'nbformat>=5,<6'])


Empezaremos importando `pandas` para cargar el archivo `blocks.csv` ubicado en la carpeta `files`. Usaremos el mismo modelo de bloques que aparece en el ejemplo de Montecarlo.

El CSV contiene el modelo disponible en el curso. No sustituimos un archivo ausente por una supuesta superficie: vamos a construir las superficies visibles a partir de sus bloques.


In [ ]:
import pandas as pd
modelo = pd.read_csv("files/blocks.csv")
print(f"Modelo disponible: {len(modelo):,} bloques")
modelo.head()


Observamos la información contenida en el modelo de bloques:

El modelo disponible contiene **119,443 bloques** y cuatro columnas:
- `X`, `Y`, `Z`: coordenadas en metros del centroide de cada bloque de 25 m de lado.
- `LITO`: litología asociada a cada bloque.

Los colores se asignan con un diccionario; no son una columna adicional del archivo. Este CSV no contiene leyes ni una evaluación económica: el visor muestra geometría y litologías, sin calcular reservas. Usaremos `unique` para observar las litologías presentes:


In [ ]:
print(modelo["LITO"].unique())

Y crearemos un diccionario asignando un color a cada litología:

In [ ]:
# Definir colores para cada litología
lito_color = {"BASAMENTO": "maroon",
              "PREMINERAL": "red",
              "PORFIDO": "blue",
              "INTRAMINERAL": "green",
              "DIQUES": "yellow",
              "CUATERNARIO": "gray"
             }

print(lito_color)

Ahora, vamos a visualizar el modelo en 3D usando **Plotly y WebGL**, disponibles en el navegador. Podremos girar, acercar y ocultar litologías desde la leyenda.

Para mantener ágil la figura, dibujaremos solo las caras que limitan con el exterior o con otra litología. Las caras entre bloques de la misma litología quedan ocultas. Conservamos todos los bloques del CSV y los contactos entre litologías, para poder observarlos al ocultar una unidad.

La lógica es: **centroides → posiciones en una malla → comparación con vecinos → caras expuestas → triángulos → figura**. No interpolamos entre bloques ni generamos roca donde no existe un bloque.


In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

# El CSV contiene centroides de bloques cúbicos de 25 m.
lado = 25.0
origen = modelo[["X", "Y", "Z"]].min().to_numpy()
indices = np.rint((modelo[["X", "Y", "Z"]].to_numpy() - origen) / lado).astype(int)
assert np.allclose(origen + indices * lado, modelo[["X", "Y", "Z"]])
assert not modelo.duplicated(["X", "Y", "Z"]).any()

# Guardar un código de litología en cada posición; -1 indica ausencia de bloque.
codigos = {lito: i for i, lito in enumerate(lito_color)}
assert modelo["LITO"].isin(codigos).all()
malla = np.full(tuple(indices.max(axis=0) + 1), -1, dtype=np.int8)
malla[tuple(indices.T)] = modelo["LITO"].map(codigos).to_numpy()

# Cuatro esquinas de cada cara, expresadas respecto al centro del cubo.
caras = [(0, 1, [[1,-1,-1], [1,1,-1], [1,1,1], [1,-1,1]]),
         (0,-1, [[-1,-1,-1], [-1,-1,1], [-1,1,1], [-1,1,-1]]),
         (1, 1, [[-1,1,-1], [-1,1,1], [1,1,1], [1,1,-1]]),
         (1,-1, [[-1,-1,-1], [1,-1,-1], [1,-1,1], [-1,-1,1]]),
         (2, 1, [[-1,-1,1], [1,-1,1], [1,1,1], [-1,1,1]]),
         (2,-1, [[-1,-1,-1], [-1,1,-1], [1,1,-1], [1,-1,-1]])]

def superficie_litologia(lito):
    ocupa = malla == codigos[lito]
    vertices = []
    for eje, sentido, esquinas in caras:
        # ¿Hay un bloque de la misma litología al otro lado de la cara?
        vecino = np.roll(ocupa, -sentido, axis=eje)
        borde = [slice(None)] * 3
        borde[eje] = -1 if sentido == 1 else 0
        vecino[tuple(borde)] = False # El extremo no se conecta con el opuesto.
        centros = origen + np.argwhere(ocupa & ~vecino) * lado
        cara = centros[:, None, :] + np.array(esquinas) * lado / 2
        vertices.append(cara.reshape(-1, 3))
    vertices = np.concatenate(vertices)

    # Cada cara cuadrada se representa con dos triángulos.
    inicio = np.arange(0, len(vertices), 4)
    i = np.concatenate([inicio, inicio])
    j = np.concatenate([inicio + 1, inicio + 2])
    k = np.concatenate([inicio + 2, inicio + 3])
    return vertices, i, j, k


In [ ]:
# Revisar la malla: el código -1 representa posiciones sin bloques.
print("Dimensiones de la malla:", malla.shape)
print("Bloques representados:", int((malla >= 0).sum()))


In [ ]:
# Una traza por litología permite ocultarla o mostrarla desde la leyenda.
fig = go.Figure()
resumen_mallas = []
for lito, color in lito_color.items():
    vertices, i, j, k = superficie_litologia(lito)
    fig.add_trace(go.Mesh3d(x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
                           i=i, j=j, k=k, name=lito, color=color,
                           flatshading=True, showlegend=True,
                           hovertemplate=lito + "<br>X=%{x} m<br>Y=%{y} m<br>Z=%{z} m<extra></extra>"))
    resumen_mallas.append({"Litología": lito,
                          "Bloques": int((modelo["LITO"] == lito).sum()),
                          "Caras visibles": len(vertices) // 4})

fig.update_layout(title=dict(text="Modelo de bloques en 3D", font=dict(size=16), x=0.02, y=0.98),
                  template="plotly_dark", height=650,
                  margin=dict(l=5, r=5, t=135, b=5),
                  legend=dict(orientation="h",
                              yanchor="bottom", y=1.02, x=0, groupclick="toggleitem"),
                  scene=dict(xaxis_title="X (m)", yaxis_title="Y (m)", zaxis_title="Z (m)",
                             aspectmode="data", camera=dict(eye=dict(x=1.6, y=-1.6, z=1.2))))
fig.show(config={"scrollZoom": True, "displaylogo": False, "responsive": True})
pd.DataFrame(resumen_mallas)


### <span style="color:gold">**Explora y explica el modelo**</span>

1. Arrastra para girar, usa la rueda para acercarte y Shift + arrastrar para desplazar. Los ejes están en metros y `aspectmode="data"` conserva sus proporciones.
2. Oculta `BASAMENTO` y `CUATERNARIO` desde la leyenda. ¿Qué contactos aparecen? Un doble clic en una entrada permite aislar una litología.
3. Explica por qué quitamos caras internas de una misma unidad, pero conservamos los contactos entre unidades diferentes. ¿Qué ocurriría al ocultar una litología si elimináramos también sus contactos?
4. El número de caras visibles no es el número de bloques. Comprueba que la suma de la columna `Bloques` coincide con `len(modelo)`.

Si pides a una IA una modificación del visor, comprueba que conserva los centroides, el lado de 25 m, las unidades, las litologías y los huecos del modelo. Una figura fluida no demuestra que la geometría sea correcta.

Podemos guardar una copia HTML interactiva para abrirla en el navegador. En JupyterLite, descarga el archivo desde el explorador de archivos para conservarlo fuera del almacenamiento del navegador.


In [ ]:
fig.write_html("modelo_bloques_3d.html", include_plotlyjs=True)


***